In [3]:
import numpy as np

# Define states and parameters
states = ['E', 'I']
obs = ['A', 'C', 'G', 'T']

# Transition probabilities
trans_probs = {
    'E': {'E': 0.9, 'I': 0.1},
    'I': {'E': 0.1, 'I': 0.9}
}

# Emission probabilities
emit_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Initial probabilities
init_probs = {'E': 0.5, 'I': 0.5}

def get_log_prob_of_a_given_path(state_path, observed_seq):
    if len(state_path) != len(observed_seq):
        raise ValueError("The state path and observed sequence must have the same length.")

    log_prob = np.log(init_probs[state_path[0]]) + np.log(emit_probs[state_path[0]][observed_seq[0]])
    for i in range(1, len(observed_seq)):
        prev_state = state_path[i - 1]
        curr_state = state_path[i]
        trans_p = trans_probs[prev_state][curr_state]
        emit_p = emit_probs[curr_state][observed_seq[i]]
        log_prob += np.log(trans_p) + np.log(emit_p)
    return round(log_prob, 2)

# Viterbi algorithm
def viterbi(obs_seq):
    if len(obs_seq) == 0:
        return (0.0, [])

    V = [{}]
    path = {}

    # Initialize base cases (t == 0)
    for state in states:
        V[0][state] = np.log(init_probs[state]) + np.log(emit_probs[state][obs_seq[0]])
        path[state] = [state]

    # Run Viterbi for t > 0
    for t in range(1, len(obs_seq)):
        V.append({})
        newpath = {}

        for curr_state in states:
            (prob, prev_state) = max(
                (V[t - 1][prev_state] + np.log(trans_probs[prev_state][curr_state]) + np.log(emit_probs[curr_state][obs_seq[t]]), prev_state)
                for prev_state in states
            )
            V[t][curr_state] = prob
            newpath[curr_state] = path[prev_state] + [curr_state]

        path = newpath

    # Get the final most probable path
    n = len(obs_seq) - 1
    (prob, state) = max((V[n][state], state) for state in states)
    return (prob, path[state])

# Example usage
state_path_example = "EEEEEEEEEEEEEEEEEEIIIIIII"
obs_seq_example = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Ensure the lengths match for testing
min_len = min(len(state_path_example), len(obs_seq_example))
state_path_example = state_path_example[:min_len]
obs_seq_example = obs_seq_example[:min_len]

log_prob = get_log_prob_of_a_given_path(state_path_example, obs_seq_example)
print(f"Log-probability of given path: {log_prob}")

viterbi_prob, viterbi_path = viterbi(obs_seq_example)
print(f"Viterbi best log-prob: {round(viterbi_prob, 2)}")
print(f"Viterbi best path: {''.join(viterbi_path)}")


Log-probability of given path: -40.95
Viterbi best log-prob: -37.88
Viterbi best path: EEEEEEEEEEEEEEEEEEEEEEEEE
